# train3: temporary training notebook (Jasper baseline against our multi-teacher recipe)

The advisor asked for a direct comparison with Jasper (Zhang et al. 2024, arXiv 2412.19048), whose loss we share (cosine x10, Gram MSE x200, ranking hinge x20, margin 0.015). Details are taken from the paper AND from the released training scripts (HF `NovaSearch/jasper_en_vision_language_v1`, `scripts/original_stella_jasper_training_codes/`; their YAML configs were not released):
- **target**: each teacher vector normalised, concatenated (here 1024 + 768 = 1792), normalised again: one target, one head (`fusion.mode=concat`); its Gram matrix is the mean of the teachers' Grams;
- **loss placement**: all terms on the head output (`loss.relation_on=head`), which Jasper also uses at inference; we put the relation terms on the pooled 768-d vector and drop the heads;
- **losses in code**: all three are batch means (the paper's sum sign is notation); the ranking hinge runs over every pair of pairs (`loss.resim_num_samples=0`; we sample 4096);
- **stages** (separate scripts: optimiser and schedule restart, lr 1e-4 / 8e-5 / 7e-5, steps 4000 / 7000 / 2200, here scaled to 30% / 53% / 17% of 5400): 1 head only; 2 head + last layer(s) + final norm; 3 adds dimension-reduction heads (independent heads on the pooled vector, initialised by averaging row groups of the full head, sim + rank only, averaged over the heads: `loss.teacher_combine=jasper`). Paper vs code: stage 2 trains the last 3 layers (paper) or the last 1 (code); stage 3 trains everything (paper) or the last 3 layers (code); embeddings are never trained in the code. Scheduler type not released: cosine with 5% warmup per stage, as in the authors' RAG-Retrieval distillation code.

Runs (1k-base student, BGE-M3 + plain 8K-embed, 200K mix v2, 5400 steps, batch 128, seed 42; reference our plain pair `scale_two_lr1e4_mix200k_s5400`, TR-MTEB 67.16, seed 43 67.25):
1. `jasper_concat_s5400`: Jasper's target and loss placement, whole encoder from the start, our single lr 1e-4 schedule (the method without the staging).
2. `jasper_code_s5400`: Jasper as in its released code (stages, layers, lr restarts, full-pair hinge, 768-d reduction head).
3. `jasper_paper_s5400`: as 2 with the paper's layer choices (stage 2: last 3 layers; stage 3: all parameters).
4. `jasper_concat_pooled_s5400`: concat target, relation terms on the pooled vector as in our recipe (isolates the single shared head).

Caveat for 2-3: Jasper's student starts from Stella, an embedding model (and one of its teachers), so training only the head and a few top layers is enough there; our student starts from a raw MLM base, so the staging can only hurt it. Run 1 is the fair test of Jasper's method on our setup, 2-3 the literal copy.

Exports after each run (`turkish_distill.export_head`): `final_head/` (backbone + pooling + the 1792-d concat head + Normalize, Jasper's inference vector) and for 2-3 `final_r768/` (the 768-d reduction head); `final/` is the pooled 768-d vector. Re-running the cells after a disconnect resumes the unfinished run and skips finished ones, then the last cell releases the runtime. Delete this notebook when the results are in.


In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)

# Runs of this notebook, one after another. (name, config, extra overrides). A later override wins.
CONFIG = 'configs/stage1_two_bge_m3_modernbert_tr_embed.yaml'   # BGE-M3 + plain 8K-embed
BASE = ["data.corpus_path=/content/drive/MyDrive/bitirme/data/mix/mix_v2_200k.jsonl", "train.max_steps=5400", "train.lr=0.0001"]
JASPER = ['fusion.mode=concat', 'loss.relation_on=head']
# Jasper's stages (4000 / 7000 / 2200 steps -> 30% / 83%), lr restarts, full-pair hinge, 768-d reduction head in stage 3
STAGED = JASPER + [
    'train.head_only_until=0.30', 'train.last_layers_until=0.83', 'train.stage_lrs=[1e-4,8e-5,7e-5]',
    'train.stage_schedule=cosine', 'train.warmup_ratio=0.05', 'loss.resim_num_samples=0', 'loss.teacher_combine=jasper',
    'fusion.reduce_dims=[768]',
]
RUNS = [
    ('jasper_concat_s5400', CONFIG, JASPER),
    ('jasper_code_s5400', CONFIG, STAGED + ['train.last_layers=1', 'train.final_stage_layers=3']),
    ('jasper_paper_s5400', CONFIG, STAGED + ['train.last_layers=3', 'train.final_stage_layers=0']),
    ('jasper_concat_pooled_s5400', CONFIG, ['fusion.mode=concat']),
]

import glob, os, shlex

for name, config, extra in RUNS:
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    if any(os.path.isdir(d + 'final') for d in run_dirs):
        print(f'\n########## {name}: already finished, skipped')
    else:
        run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
        overrides = ' '.join(f'--override {shlex.quote(o)}' for o in [f'name={name}'] + BASE + extra)
        print(f'\n########## {name} {run_id_arg or "(new run)"}')
        get_ipython().system(f'python -m turkish_distill.train --config {config} {run_id_arg} {overrides}')
    # Jasper evaluates its head outputs: export backbone + pooling + head (+ Normalize) next to final/
    for d in sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/final')):
        heads = [('concat', '_head')] + ([('concat_r768', '_r768')] if 'fusion.reduce_dims=[768]' in extra else [])
        for head, suffix in heads:
            if not os.path.isdir(d + suffix):
                get_ipython().system(f'python -m turkish_distill.export_head --model {d} --head {head} --out {d}{suffix}')

print('\nEval presets (paste into an eval notebook):')
for name, config, extra in RUNS:
    for d in sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/final')):
        run = os.path.basename(os.path.dirname(d))
        for suffix in ('', '_head', '_r768'):
            if os.path.isdir(f'{DRIVE_ROOT}/runs/{name}/{run}/final{suffix}'):
                print(f"    'd_{name}{suffix}': dict(model=f'{{DRIVE_ROOT}}/runs/{name}/{run}/final{suffix}'),")


In [ ]:
from google.colab import runtime
runtime.unassign()